# Trace one bid end-to-end — a `DETAILBOQAVAILABLE = 'Y'` tender

Unity Catalog: `ingestion_framework_test.bid_data_exploration`

Run 2 (`01_rfq.ipynb`) found `DETAILBOQAVAILABLE = 'Y'` on only 412 of 30,338 RFQs. Rather than judge that flag from aggregate counts alone, this notebook picks one real `Y`-flagged tender and follows it through every table — `rfq` → `rfqvendor` → `quotationline` → `altquotationline` → `docinfo`/`doclinks`/`vw_rfqvendor_documents` — to see what "has a detailed BOQ" actually looks like end to end. Contrast with `08_trace_bid_without_boq.ipynb` (D-111808, plus a `null`-flagged example).

## Step 1 — find a real candidate
Not just any `Y`-flagged RFQ — one with substantial `quotationline` data, so there's actually something to trace. Ordered by line count so the richest example surfaces first.

In [ ]:
%sql
SELECT r.RFQNUM, r.DESCRIPTION, r.ORGID, r.ENTERDATE,
       COUNT(ql.QUOTATIONLINEID) AS line_count,
       COUNT(DISTINCT ql.VENDOR) AS vendor_count,
       COUNT(DISTINCT ql.BOQITEMNUM) AS distinct_boqitems
FROM ingestion_framework_test.bid_data_exploration.rfq r
LEFT JOIN ingestion_framework_test.bid_data_exploration.quotationline ql ON r.RFQNUM = ql.RFQNUM
WHERE r.DETAILBOQAVAILABLE = 'Y'
GROUP BY r.RFQNUM, r.DESCRIPTION, r.ORGID, r.ENTERDATE
ORDER BY line_count DESC
LIMIT 20

## Step 2 — set the RFQNUM to trace
Copy an `RFQNUM` from the candidates above into the widget this next cell creates (it'll appear at the top of the notebook), then run every cell below.

In [ ]:
dbutils.widgets.text("rfqnum", "", "RFQNUM to trace")

### `rfq` — header

In [ ]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.rfq WHERE RFQNUM = :rfqnum

### `rfqvendor` — invited vendors

In [ ]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.rfqvendor WHERE RFQNUM = :rfqnum ORDER BY VENDOR

### `quotationline` — the itemized BOQ itself (the whole point of this notebook)

In [ ]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.quotationline WHERE RFQNUM = :rfqnum ORDER BY VENDOR, BOQITEMNUM, RFQLINENUM

### `altquotationline` — any alternates offered

In [ ]:
%sql
SELECT * FROM ingestion_framework_test.bid_data_exploration.altquotationline WHERE RFQNUM = :rfqnum ORDER BY VENDOR, RFQLINENUM

### `docinfo` / `doclinks` / `vw_rfqvendor_documents` — attached documents

Notebook 05 is still deliberately deferred as a general investigation, but it's worth checking directly here for one specific tender: does a real bid document exist and link back to this RFQNUM? Schemas are unknown so far (notebook 05 was never run) — start with `DESCRIBE`.

In [ ]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.docinfo

In [ ]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.doclinks

In [ ]:
%sql
DESCRIBE TABLE ingestion_framework_test.bid_data_exploration.vw_rfqvendor_documents

Once the schemas above are visible, uncomment and fix the column name below (guessing `RFQNUM`, following the convention every other table uses — may need correcting):

In [ ]:
%sql
-- SELECT * FROM ingestion_framework_test.bid_data_exploration.vw_rfqvendor_documents WHERE RFQNUM = :rfqnum

**Observations:**
- _(fill in: which RFQNUM did you trace, and what does a genuinely populated BOQ look like — is `BOQITEMNUM` a real per-item code here, or still a coarse section label like the Run 2 examples? Does `LINETYPE` split CIF/Erection for this one? Are documents actually attached and retrievable?)_